# PCA — Principal Component Analysis
**Mathematics for Machine Learning, Ch 10**

This notebook implements PCA from scratch and verifies against scikit-learn.
Topics covered:
1. Data centering
2. Covariance matrix and eigendecomposition
3. PCA from scratch + sklearn verification
4. Explained variance and scree plot
5. 2D → 1D projection, reconstruction, and reconstruction error
6. High-dimensional PCA (N < D trick)
7. Image compression (synthetic rank-5 image)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

---
## 1. Data Centering

PCA always begins with centering: subtract the sample mean $\mu = \frac{1}{N}\sum_n x_n$ from every data point.
Without centering the first principal component often points toward the data centroid rather than capturing variance structure.

We generate 2D correlated data and show before/after centering.

In [ ]:
# Generate 2D correlated data
mean = np.array([3.0, 5.0])
cov  = np.array([[2.0, 1.5],
                 [1.5, 2.0]])
N = 200
X = np.random.multivariate_normal(mean, cov, size=N)  # (N, 2)

# Center
mu   = X.mean(axis=0)
X_c  = X - mu  # centered data

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, data, title, color in zip(
    axes,
    [X, X_c],
    ['Original data (uncentered)', 'Centered data (mean subtracted)'],
    ['steelblue', 'tomato']
):
    ax.scatter(data[:, 0], data[:, 1], alpha=0.4, s=18, color=color)
    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.axvline(0, color='k', lw=0.7, ls='--')
    ax.scatter(*data.mean(axis=0), color='black', s=80, zorder=5, label=f'mean={data.mean(axis=0).round(2)}')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_aspect('equal')

plt.suptitle('Step 1: Data Centering', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Original mean: {mu.round(3)}')
print(f'Centered mean: {X_c.mean(axis=0).round(6)}')

---
## 2. Covariance Matrix and Eigendecomposition

The **data covariance matrix** is:
$$S = \frac{1}{N} \tilde{X}^\top \tilde{X} \in \mathbb{R}^{D \times D}$$

$S$ is symmetric positive semidefinite. By the spectral theorem: $S = U \Lambda U^\top$,
where the columns of $U$ are the **principal components** (eigenvectors of $S$) and
$\Lambda = \text{diag}(\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_D)$ contains the eigenvalues.

The **variance captured along principal component $b_i$** is exactly $\lambda_i$.

In [ ]:
# Compute sample covariance matrix
S = (X_c.T @ X_c) / N  # (D, D)
print('Sample covariance matrix S:')
print(S.round(4))

# Eigendecomposition (numpy returns eigenvalues not necessarily sorted)
eigenvalues, eigenvectors = np.linalg.eigh(S)  # eigh for symmetric matrices

# Sort descending
idx         = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]  # columns are eigenvectors

print(f'\nEigenvalues (descending): {eigenvalues.round(4)}')
print(f'Eigenvectors (columns):\n{eigenvectors.round(4)}')
print(f'\nTotal variance tr(S) = {np.trace(S):.4f} == sum of eigenvalues = {eigenvalues.sum():.4f}')

# Visualize: scatter + principal axes
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X_c[:, 0], X_c[:, 1], alpha=0.35, s=18, color='steelblue', label='centered data')

colors = ['crimson', 'darkorange']
for i, (val, vec, col) in enumerate(zip(eigenvalues, eigenvectors.T, colors)):
    scale = np.sqrt(val) * 2  # arrow length proportional to std dev
    ax.annotate('', xy=scale * vec, xytext=-scale * vec,
                arrowprops=dict(arrowstyle='<->', color=col, lw=2.5))
    ax.text(*(scale * vec * 1.15), f'PC{i+1}\n$\\lambda_{i+1}$={val:.2f}',
            color=col, ha='center', fontsize=10, fontweight='bold')

ax.axhline(0, color='k', lw=0.5, ls='--')
ax.axvline(0, color='k', lw=0.5, ls='--')
ax.set_aspect('equal')
ax.set_title('Principal axes overlaid on centered data')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. PCA From Scratch — and Verification Against sklearn

Full PCA pipeline:
1. Center: $\tilde{X} = X - \mathbf{1}\mu^\top$
2. Covariance: $S = \frac{1}{N}\tilde{X}^\top\tilde{X}$
3. Eigendecompose: $S = U\Lambda U^\top$
4. Choose top $M$ components: $U_M = U[:, 1:M]$
5. Project: $Z = \tilde{X} U_M$
6. Reconstruct: $\hat{\tilde{X}} = Z U_M^\top$

In [ ]:
def pca_from_scratch(X, n_components):
    """PCA implemented from the MML Ch10 algorithm."""
    N = X.shape[0]
    # Step 1: center
    mu   = X.mean(axis=0)
    X_c  = X - mu
    # Step 2: covariance
    S    = (X_c.T @ X_c) / N
    # Step 3: eigendecompose
    vals, vecs = np.linalg.eigh(S)
    idx  = np.argsort(vals)[::-1]
    vals = vals[idx]
    vecs = vecs[:, idx]
    # Step 4: select top M
    U_M  = vecs[:, :n_components]   # (D, M)
    # Step 5: project
    Z    = X_c @ U_M                # (N, M)
    # Step 6: reconstruct
    X_rec = Z @ U_M.T + mu         # (N, D)
    return {
        'mu': mu, 'components': U_M, 'eigenvalues': vals,
        'Z': Z, 'X_reconstructed': X_rec, 'S': S
    }


# Our scratch PCA with M=1 component
result_scratch = pca_from_scratch(X, n_components=1)

# sklearn PCA
pca_sk = PCA(n_components=1)
pca_sk.fit(X)

# Compare eigenvectors (up to sign flip — both are equally valid)
pc_scratch = result_scratch['components'][:, 0]
pc_sklearn  = pca_sk.components_[0]

# Align signs
if np.dot(pc_scratch, pc_sklearn) < 0:
    pc_scratch = -pc_scratch

print('=== Verification Against sklearn ===')
print(f'Scratch PC1:  {pc_scratch.round(6)}')
print(f'sklearn PC1:  {pc_sklearn.round(6)}')
print(f'Max absolute difference: {np.abs(pc_scratch - pc_sklearn).max():.2e}')

ev_scratch = result_scratch['eigenvalues'][0]
ev_sklearn  = pca_sk.explained_variance_[0]
print(f'\nScratch eigenvalue: {ev_scratch:.6f}')
print(f'sklearn eigenvalue: {ev_sklearn:.6f}  (sklearn uses 1/(N-1), so will differ slightly)')
print(f'Ratio N/(N-1): {N/(N-1):.6f}  — accounts for the difference')

---
## 4. Explained Variance — Scree Plot

The **explained variance ratio** of the first $M$ components:
$$\text{EVR}(M) = \frac{\lambda_1 + \cdots + \lambda_M}{\lambda_1 + \cdots + \lambda_D} = \frac{\sum_{i=1}^M \sigma_i^2}{\|\tilde{X}\|_F^2}$$

The **scree plot** shows $\lambda_i$ vs. component index. Look for the "elbow" — where the curve flattens.

In [ ]:
# Generate higher-dimensional data to make a scree plot meaningful
D = 10
N2 = 300
# True rank-3 signal + noise
true_comps = np.random.randn(D, 3)
scores     = np.random.randn(N2, 3) * np.array([5, 2, 1])
noise      = np.random.randn(N2, D) * 0.3
X_hd       = scores @ true_comps.T + noise

pca_hd = pca_from_scratch(X_hd, n_components=D)
evals  = pca_hd['eigenvalues']
total  = evals.sum()
evr    = evals / total
cumevr = np.cumsum(evr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
ax = axes[0]
ax.bar(range(1, D + 1), evals, color='steelblue', alpha=0.8, edgecolor='navy')
ax.plot(range(1, D + 1), evals, 'o-', color='navy', lw=1.5)
ax.set_xlabel('Component index')
ax.set_ylabel('Eigenvalue (variance)')
ax.set_title('Scree plot — eigenvalue spectrum')
ax.set_xticks(range(1, D + 1))

# Cumulative variance
ax = axes[1]
ax.plot(range(1, D + 1), cumevr * 100, 's-', color='tomato', lw=2)
ax.axhline(95, color='gray', ls='--', lw=1.2, label='95% threshold')
ax.fill_between(range(1, D + 1), cumevr * 100, alpha=0.15, color='tomato')
ax.set_xlabel('Number of components M')
ax.set_ylabel('Cumulative explained variance (%)')
ax.set_title('Cumulative explained variance')
ax.set_xticks(range(1, D + 1))
ax.legend()

# Annotate which M reaches 95%
m_95 = np.argmax(cumevr >= 0.95) + 1
ax.axvline(m_95, color='gray', ls=':', lw=1.2)
ax.text(m_95 + 0.1, 50, f'M={m_95}\nreaches 95%', fontsize=9)

plt.suptitle('Scree Plot and Cumulative Explained Variance', fontweight='bold')
plt.tight_layout()
plt.show()

print('Individual explained variance ratios:')
for i, (e, r) in enumerate(zip(evals, evr)):
    print(f'  PC{i+1}: λ={e:.4f}  EVR={r*100:.1f}%  cumulative={cumevr[i]*100:.1f}%')

---
## 5. 2D → 1D Projection: Reconstruction and Error

With $M=1$ component:
- **Projection scores**: $z_n = b_1^\top \tilde{x}_n \in \mathbb{R}$ (scalar)
- **Reconstruction**: $\hat{\tilde{x}}_n = z_n b_1 \in \mathbb{R}^2$ (back in 2D, on the PC1 line)
- **Reconstruction error**: $J = \frac{1}{N}\sum_n \|\tilde{x}_n - \hat{\tilde{x}}_n\|^2 = \lambda_2$ (discarded eigenvalue)

In [ ]:
result_1d = pca_from_scratch(X, n_components=1)
X_rec_1d  = result_1d['X_reconstructed']  # (N, 2) — reconstructed in original space
b1        = result_1d['components'][:, 0]  # first principal component

# Reconstruction error
errors    = np.linalg.norm(X - X_rec_1d, axis=1)
J_empirical = np.mean(errors**2)
J_theory    = result_1d['eigenvalues'][1]  # should equal λ₂

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: scatter with PC1 axis and reconstructed points
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], alpha=0.3, s=18, color='steelblue', label='original')
ax.scatter(X_rec_1d[:, 0], X_rec_1d[:, 1], alpha=0.5, s=12, color='tomato', label='reconstructed')

# Draw projection lines for a subset
for i in range(0, N, 10):
    ax.plot([X[i, 0], X_rec_1d[i, 0]], [X[i, 1], X_rec_1d[i, 1]],
            color='gray', lw=0.6, alpha=0.6)

# Draw PC1 axis
t = np.linspace(-4, 4, 100)
pc1_line = np.outer(t, b1) + result_1d['mu']
ax.plot(pc1_line[:, 0], pc1_line[:, 1], 'k-', lw=1.5, label='PC1 axis')

ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.set_title('Original, reconstructed, and projection lines')

# Right: histogram of reconstruction errors
ax = axes[1]
ax.hist(errors, bins=25, color='steelblue', edgecolor='navy', alpha=0.8)
ax.axvline(errors.mean(), color='tomato', ls='--', lw=2, label=f'mean error = {errors.mean():.3f}')
ax.set_xlabel('Reconstruction error ||x - x_hat||')
ax.set_ylabel('Count')
ax.set_title('Distribution of reconstruction errors')
ax.legend()

plt.suptitle('2D → 1D Projection: Reconstruction and Error', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Empirical J = (1/N) Σ ||x - x̂||² = {J_empirical:.6f}')
print(f'Theoretical J = λ₂                 = {J_theory:.6f}')
print(f'Match: {np.isclose(J_empirical, J_theory, atol=1e-4)}')
print(f'\nExplained variance ratio (M=1): {result_1d["eigenvalues"][0]/result_1d["eigenvalues"].sum()*100:.1f}%')

---
## 6. PCA in High Dimensions (N < D trick)

When $N < D$, computing $S = \frac{1}{N}\tilde{X}^\top\tilde{X} \in \mathbb{R}^{D \times D}$ is infeasible.
Instead, compute the $N \times N$ matrix:
$$C = \frac{1}{N}\tilde{X}\tilde{X}^\top \in \mathbb{R}^{N \times N}$$

$C$ and $S$ share the same nonzero eigenvalues. The principal components of $S$ are recovered as:
$$u = \frac{1}{\sqrt{N\lambda}} \tilde{X}^\top v$$
where $Cv = \lambda v$.

In [ ]:
# Setup: N < D scenario
N_small = 50
D_large = 200

# Generate data with low intrinsic rank
true_rank = 5
A = np.random.randn(N_small, true_rank)
B = np.random.randn(true_rank, D_large)
X_hd = A @ B + 0.1 * np.random.randn(N_small, D_large)  # (N_small, D_large)

# Center
mu_hd = X_hd.mean(axis=0)
X_hd_c = X_hd - mu_hd

# === Method 1: Standard (D×D) — only feasible here for verification ===
S_DD = (X_hd_c.T @ X_hd_c) / N_small              # (D, D) — expensive
evals_S, evecs_S = np.linalg.eigh(S_DD)
idx_S = np.argsort(evals_S)[::-1]
evals_S = evals_S[idx_S]
evecs_S = evecs_S[:, idx_S]  # principal components — (D, D)

# === Method 2: High-dim (N×N) ===
C_NN = (X_hd_c @ X_hd_c.T) / N_small              # (N, N) — cheap
evals_C, evecs_C = np.linalg.eigh(C_NN)
idx_C = np.argsort(evals_C)[::-1]
evals_C = evals_C[idx_C]
evecs_C = evecs_C[:, idx_C]

# Recover principal components of S from eigenvectors of C
M = 5  # top components to recover
pcs_from_C = np.zeros((D_large, M))
for i in range(M):
    lam = evals_C[i]
    v   = evecs_C[:, i]
    if lam > 1e-10:
        u = X_hd_c.T @ v / np.sqrt(N_small * lam)
        pcs_from_C[:, i] = u

# Compare: cosine similarity between recovered and ground-truth PCs
print('=== High-Dim PCA Verification ===')
print(f'Data shape: N={N_small}, D={D_large}  (N < D)')
print(f'C is ({N_small}×{N_small}), S would be ({D_large}×{D_large})')
print()
print('Cosine similarity between PC from N×N trick vs D×D eigendecomp:')
for i in range(M):
    cos_sim = abs(np.dot(pcs_from_C[:, i], evecs_S[:, i]))
    print(f'  PC{i+1}: |cos θ| = {cos_sim:.8f}  (should be ≈ 1.0)')

# Eigenvalue comparison
print()
print('Eigenvalue comparison (N×N trick vs D×D eigendecomp):')
for i in range(M):
    print(f'  PC{i+1}: λ_C={evals_C[i]:.6f}  λ_S={evals_S[i]:.6f}')

---
## 7. Image Compression (Synthetic Rank-5 Image)

We create a synthetic 20×20 grayscale image of rank 5, then show reconstructions
at ranks 1, 3, 5, and 10 using PCA (equivalent to truncated SVD via the Eckart-Young theorem).

The Frobenius reconstruction error is:
$$\|\tilde{X} - \hat{\tilde{X}}\|_F^2 = \sum_{i=M+1}^{D} \sigma_i^2$$

In [ ]:
# Create a synthetic 20×20 rank-5 "image"
H, W = 20, 20
rank_true = 5
np.random.seed(7)

# Rank-5 image = sum of 5 outer products
U_true = np.random.randn(H, rank_true)
V_true = np.random.randn(W, rank_true)
svals  = np.array([20, 10, 5, 2, 1])  # singular values
img    = (U_true * svals) @ V_true.T  # (H, W)
noise  = np.random.randn(H, W) * 0.1
img   += noise

# Treat each row as a data point (N=H, D=W)
# Center (PCA centers along columns)
img_mean = img.mean(axis=0)
img_c    = img - img_mean

# SVD of centered image
U_svd, s_svd, Vt_svd = np.linalg.svd(img_c, full_matrices=False)

ranks_to_show = [1, 3, 5, 10]
reconstructions = {}
errors = {}

for r in ranks_to_show:
    r_eff = min(r, len(s_svd))
    img_rec = (U_svd[:, :r_eff] * s_svd[:r_eff]) @ Vt_svd[:r_eff, :] + img_mean
    reconstructions[r] = img_rec
    frob_err = np.sum(s_svd[r_eff:]**2)
    errors[r] = frob_err

# Plot
fig, axes = plt.subplots(1, len(ranks_to_show) + 1, figsize=(14, 3.5))

vmin, vmax = img.min(), img.max()

axes[0].imshow(img, cmap='viridis', vmin=vmin, vmax=vmax)
axes[0].set_title('Original\n(rank 5 + noise)', fontsize=10)
axes[0].axis('off')

for ax, r in zip(axes[1:], ranks_to_show):
    ax.imshow(reconstructions[r], cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_title(f'Rank {r}\nFrob err={errors[r]:.1f}', fontsize=10)
    ax.axis('off')

plt.suptitle('Image Compression via PCA / Truncated SVD (20×20 synthetic rank-5 image)',
             fontweight='bold')
plt.tight_layout()
plt.show()

# Compression ratio
print('Singular values:', s_svd.round(2))
total_sv2 = (s_svd**2).sum()
print()
for r in ranks_to_show:
    r_eff = min(r, len(s_svd))
    captured = (s_svd[:r_eff]**2).sum() / total_sv2 * 100
    stored   = H * r_eff + r_eff + W * r_eff  # U cols + s values + V rows
    original = H * W
    print(f'Rank {r:2d}: captures {captured:.1f}% variance | '
          f'stores {stored} vs {original} values (ratio {original/stored:.1f}x)')

---
## Summary

| Concept | Key formula |
|---------|-------------|
| Center | $\tilde{x}_n = x_n - \mu$ |
| Covariance | $S = \frac{1}{N}\tilde{X}^\top\tilde{X}$ |
| Variance along $b$ | $\text{Var} = b^\top S b$ |
| PCA condition | $Sb = \lambda b$ (eigenvector) |
| Total variance | $\text{tr}(S) = \sum_i \lambda_i$ |
| Reconstruction error | $J = \sum_{i>M} \lambda_i$ |
| High-dim matrix | $C = \frac{1}{N}\tilde{X}\tilde{X}^\top \in \mathbb{R}^{N \times N}$ |
| Frobenius error | $\|\tilde{X} - \hat{\tilde{X}}\|_F^2 = \sum_{i>M} \sigma_i^2$ |